# Computer vision project: 2D grasp prediction

**Aim:** From an RGB image and object mask, predict where a two-finger gripper should grasp and at what angle/opening. Compare a geometry baseline, a CNN, the hybrid score, and a robustness-aware hybrid extension. The small Python files contain reusable functions; this notebook follows the same load → display → model → train → evaluate order as our class exercises.


## 1. Imports and settings

`DATA_FOLDER` points to the provided 3.4 GB Jacquard subset. Use a new `RUN_FOLDER` for each trial or full training run so saved results do not get mixed together.


In [ ]:
from pathlib import Path
import json
import random
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader

from data import find_images, split_by_object, choose, read_scene, make_input, make_targets, JacquardDataset
from model import GraspNet, grasp_loss, load_checkpoint
from methods import predict_three, corners, rectangle_iou, angle_error, matches_annotation, draw_grasp

DATA_FOLDER = Path('C:/Users/krish/Desktop/Jacquard_Dataset_11')
# Keep DATA_FOLDER unchanged for fair comparisons. Put newly downloaded objects here.
EXTRA_TRAIN_FOLDERS = [
    Path('C:/Users/krish/Desktop/cvl project_robust mode/more data')
]
RUN_FOLDER = Path('runs_more_data')
SIZE = 128
SEED = 42
QUICK_RUN = False
assert DATA_FOLDER.exists(), 'Set DATA_FOLDER to your extracted Jacquard folder'
RUN_FOLDER.mkdir(exist_ok=True)


## 2. Read and display one training scene

Jacquard provides an RGB image, a binary mask, and grasp rectangles in a matching text file. Each grasp has `(x, y, angle, opening, jaw size)`. The yellow rectangles below are some valid annotated grasps.


In [ ]:
all_images = find_images(DATA_FOLDER)
print('Scenes:', len(all_images), '| Objects:', len({p.parent for p in all_images}))
rgb, mask, grasps = read_scene(all_images[0])
print('Image:', rgb.shape, '| Valid grasps:', len(grasps))
shown = rgb.copy()
for grasp in grasps[:15]:
    cv2.polylines(shown, [corners(grasp).astype(np.int32)], True, (255, 255, 0), 2)
fig, ax = plt.subplots(1, 3, figsize=(14, 4))
ax[0].imshow(rgb); ax[0].set_title('RGB')
ax[1].imshow(mask, cmap='gray'); ax[1].set_title('Object mask')
ax[2].imshow(shown); ax[2].set_title('Annotated grasps')
for item in ax: item.axis('off')
plt.show()


## 3. Split by object and make one training target

Different views of one object must stay in the same train/validation/test group. Keep the original subset fixed; any folders in `EXTRA_TRAIN_FOLDERS` add only new object IDs to training, so the existing validation and test scenes remain comparable. For a large subset, `QUICK_RUN=True` samples at most 200 training, 50 validation, and 50 test images **after** that object split. The CNN targets are maps: grasp quality, `cos(2 angle)`, `sin(2 angle)`, and opening width.


In [ ]:
groups = split_by_object(all_images, seed=SEED)
original_object_ids = {path.parent.name for path in all_images}
for extra_folder in EXTRA_TRAIN_FOLDERS:
    for path in find_images(extra_folder):
        if path.parent.name not in original_object_ids:
            groups['train'].append(path)
train_images = choose(groups['train'], 200 if QUICK_RUN else None, SEED)
val_images = choose(groups['val'], 50 if QUICK_RUN else None, SEED + 1)
test_images = choose(groups['test'], 50 if QUICK_RUN else None, SEED + 2)
print('Train/validation/test scenes:', len(train_images), len(val_images), len(test_images))
(RUN_FOLDER / 'split.json').write_text(json.dumps(
    {name: sorted({str(p.parent) for p in files}) for name, files in groups.items()}, indent=2))

image = make_input(rgb, mask, SIZE)
quality, regression, valid = make_targets(grasps, mask.shape, SIZE)
print('CNN input:', image.shape, '| Target quality:', quality.shape)
plt.imshow(quality, cmap='hot'); plt.title('Bright spots = good grasp centers'); plt.axis('off'); plt.show()


## 4. Build and train the CNN

`GraspNet` takes RGB + mask (4 channels) and predicts 4 maps at every pixel. Training images and grasp labels may be flipped together; validation and test scenes are never augmented. Quick mode trains for 3 epochs; full mode trains for 12. The best validation checkpoint becomes `best.pt`.


In [ ]:
torch.manual_seed(SEED)
random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
net = GraspNet().to(device)
print('Device:', device, '| Parameters:', sum(p.numel() for p in net.parameters()))
train_loader = DataLoader(JacquardDataset(train_images, SIZE, augment=True), batch_size=4, shuffle=True, num_workers=0)
val_loader = DataLoader(JacquardDataset(val_images, SIZE), batch_size=4, num_workers=0)
optimizer = torch.optim.Adam(net.parameters(), lr=0.001)
EPOCHS = 8 if len(all_images) <= 60 else (3 if QUICK_RUN else 12)
(RUN_FOLDER / 'run_info.json').write_text(json.dumps({
    'device': str(device), 'torch': torch.__version__,
    'train_scenes': len(train_images), 'validation_scenes': len(val_images),
    'test_scenes': len(test_images), 'seed': SEED, 'epochs': EPOCHS,
    'augmentation': 'horizontal and vertical flips on training scenes only'}, indent=2))
history, best_loss = [], float('inf')

for epoch in range(1, EPOCHS + 1):
    net.train()
    train_losses = []
    for image, quality, regression, valid in train_loader:
        image, quality, regression, valid = [item.to(device) for item in (image, quality, regression, valid)]
        optimizer.zero_grad()
        loss = grasp_loss(net(image), quality, regression, valid)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())
    net.eval()
    with torch.no_grad():
        val_losses = [grasp_loss(net(x.to(device)), q.to(device), r.to(device), v.to(device)).item()
                      for x, q, r, v in val_loader]
    train_loss, val_loss = float(np.mean(train_losses)), float(np.mean(val_losses))
    history.append({'epoch': epoch, 'train': train_loss, 'validation': val_loss})
    print(f'Epoch {epoch}: train={train_loss:.3f}, validation={val_loss:.3f}')
    if val_loss < best_loss:
        best_loss = val_loss
        torch.save({'model': net.state_dict(), 'image_size': SIZE}, RUN_FOLDER / 'best.pt')
(RUN_FOLDER / 'history.json').write_text(json.dumps(history, indent=2))


## 5. Plot the learning curve

The model trains on training objects; validation objects are only used to choose the best epoch. A falling training loss alone does not prove generalization.


In [ ]:
plt.plot([row['epoch'] for row in history], [row['train'] for row in history], 'o-', label='train')
plt.plot([row['epoch'] for row in history], [row['validation'] for row in history], 'o-', label='validation')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.grid(alpha=.3); plt.legend(); plt.show()


## 6. Evaluate the four methods on held-out objects

The CNN proposes candidate grasp rectangles. Geometry-only uses the mask centroid and short PCA axis. Hybrid reranks CNN candidates with mask-based geometric suitability; robust hybrid also tests small pose errors. The CNN now has training-only flip augmentation. Rectangle thickness is a display/evaluation estimate equal to 50% of opening, chosen from training and validation data; it does not change the gripper opening. A match requires angle error at most 30° and rotated rectangle IoU at least 0.25 against any annotation. This is **annotation agreement**, not a physical grasp trial.


In [ ]:
trained, size, device = load_checkpoint(RUN_FOLDER / 'best.pt')
results = {name: {'matched': [], 'iou': [], 'angle_error': []}
           for name in ('geometry', 'learned', 'hybrid', 'robust')}
for path in test_images:
    rgb, mask, truths = read_scene(path)
    predictions = predict_three(trained, size, device, rgb, mask, include_robust=True)
    for name, grasp in predictions.items():
        results[name]['matched'].append(matches_annotation(grasp, truths))
        if grasp is not None and truths:
            closest = max(truths, key=lambda truth: rectangle_iou(grasp, truth))
            results[name]['iou'].append(rectangle_iou(grasp, closest))
            results[name]['angle_error'].append(np.degrees(angle_error(grasp.angle, closest.angle)))

summary = {name: {'match_fraction': float(np.mean(values['matched'])),
                  'mean_iou': float(np.mean(values['iou'])) if values['iou'] else None,
                  'mean_angle_error_deg': float(np.mean(values['angle_error'])) if values['angle_error'] else None}
           for name, values in results.items()}
print(json.dumps(summary, indent=2))
(RUN_FOLDER / 'metrics.json').write_text(json.dumps(summary, indent=2))


## 7. Visual comparison on one held-out image

The red dot is the grasp center. The yellow rectangle indicates the gripper pose. Opening width is in image pixels, not centimeters.


In [ ]:
rgb, mask, _ = read_scene(test_images[0])
predictions = predict_three(trained, size, device, rgb, mask, include_robust=True)
panels = [draw_grasp(rgb, mask, predictions[name], name)
          for name in ('geometry', 'learned', 'hybrid', 'robust')]
cv2.imwrite(str(RUN_FOLDER / 'comparison.png'), cv2.hconcat(panels))
fig, ax = plt.subplots(1, 4, figsize=(18, 5))
for item, panel, name in zip(ax, panels, predictions):
    item.imshow(cv2.cvtColor(panel, cv2.COLOR_BGR2RGB))
    item.set_title(name); item.axis('off')
plt.show()


## 8. Check robustness on one held-out scene

Robust mode tests eight small errors around each CNN candidate, averages their hybrid scores, and uses `0.7 × normal hybrid + 0.3 × average perturbed score` to pick a grasp. The current hybrid score uses 90% CNN quality and 10% geometric suitability, chosen on validation scenes. This measures sensitivity to those specific errors; it is not a physical grasp trial.


In [ ]:
rgb, mask, _ = read_scene(test_images[0])
predictions = predict_three(trained, size, device, rgb, mask, include_robust=True)
robust_grasp = predictions['robust']
if robust_grasp is not None:
    print('Normal hybrid:', round(robust_grasp.hybrid_score, 3))
    print('Average perturbed:', round(robust_grasp.robustness_score, 3))
    print('Worst perturbation:', round(robust_grasp.worst_case_score, 3))
    print('Final:', round(robust_grasp.final_score, 3))
panels = [draw_grasp(rgb, mask, predictions['hybrid'], 'hybrid'),
          draw_grasp(rgb, mask, robust_grasp, 'robust', show_robustness=True)]
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
for item, panel in zip(ax, panels):
    item.imshow(cv2.cvtColor(panel, cv2.COLOR_BGR2RGB)); item.axis('off')
plt.show()


## 9. Local laptop webcam demo

Once `best.pt` exists, run **locally** from this project folder:

```powershell
& "C:\Users\krish\Documents\Codex\2026-09-28\o\work\grasp_gpu_venv\Scripts\python.exe" .\webcam.py .\runs_more_data\best.pt
```

Show an empty table and press **b** to capture its background. Add one object; the app builds a mask from the changed pixels. Press **g** for geometry, **l** for learning, **h** for hybrid, **r** for robust hybrid, and **q** to quit. The webcam needs a stationary camera and stable lighting. We do not convert pixels to physical gripper units or control a robot.
